# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [16]:
# ML-07: inspect the available FlyRank data and schema
import os
import glob
import pandas as pd

REPO = "/content/flyrank-ml-internship-starter"

if not os.path.exists(REPO):
    !git clone -q https://github.com/Sameer99-star/flyrank-ml-internship-starter.git

print("Repository:", REPO)

files = []
for root, dirs, filenames in os.walk(f"{REPO}/data"):
    for name in filenames:
        files.append(os.path.join(root, name))

print("\nData files found:")
for f in files:
    print(" -", os.path.relpath(f, REPO))

print("\nCandidate tabular files:")
for f in files:
    if f.lower().endswith((".csv", ".parquet", ".json")):
        print(" -", os.path.relpath(f, REPO))

Repository: /content/flyrank-ml-internship-starter

Data files found:
 - data/raw/content_refresh_anonymized.csv

Candidate tabular files:
 - data/raw/content_refresh_anonymized.csv


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
# ML-07 — Build the ranked action queue
# Rule: prioritize content that is both older and declining.

import os
import pandas as pd

work_outputs = f"{REPO}/work/outputs"
os.makedirs(work_outputs, exist_ok=True)

ranked = df.copy()

# Score based only on currently observed signals.
# Higher score = stronger refresh candidate.
ranked["score"] = (
    ranked["days_since_last_update"].fillna(0).clip(lower=0) / 30
    + ranked["trend_pct"].fillna(0).clip(upper=0).abs() / 10
)

# One reason code for the rule.
ranked["reason_code"] = "AGE_AND_DECLINE"

# Action label.
ranked["action"] = "REFRESH"

# Rank highest-priority candidates first.
ranked = ranked.sort_values(
    ["score", "days_since_last_update"],
    ascending=[False, False]
).reset_index(drop=True)

ranked["rank"] = ranked.index + 1

# Keep the ranked queue compact and useful.
output_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "freshness_tier",
    "days_since_last_update",
    "trend_direction",
    "trend_pct",
    "ctr",
    "avg_position"
]

baseline_queue = ranked[output_columns]

output_path = f"{work_outputs}/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

print("Rule: prioritize content that is both older and declining.")
print("Reason code: AGE_AND_DECLINE")
print("Action: REFRESH")
print(f"\nRows ranked: {len(baseline_queue)}")
print(f"CSV written to: {output_path}")

display(baseline_queue.head(10))

Rule: prioritize content that is both older and declining.
Reason code: AGE_AND_DECLINE
Action: REFRESH

Rows ranked: 30000
CSV written to: /content/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv


,rank,content_id,score,reason_code,action,freshness_tier,days_since_last_update,trend_direction,trend_pct,ctr,avg_position
0,1,content_f6fdf87348f6,22.433333,AGE_AND_DECLINE,REFRESH,181+,373,down,-100.0,0.00,32.5
1,2,content_1b4ec72dafd4,22.400000,AGE_AND_DECLINE,REFRESH,181+,372,down,-100.0,0.00,7.0
2,3,content_55a5b1c46474,21.283333,AGE_AND_DECLINE,REFRESH,181+,373,down,-88.5,0.00,7.5
3,4,content_7a888d3d99c8,20.433333,AGE_AND_DECLINE,REFRESH,181+,313,down,-100.0,0.00,67.6
4,5,content_94991fe6268c,20.433333,AGE_AND_DECLINE,REFRESH,181+,313,down,-100.0,0.00,12.4
5,6,content_ab18b5811c02,20.166667,AGE_AND_DECLINE,REFRESH,181+,305,down,-100.0,0.00,29.8
6,7,content_84d12054c0c0,20.133333,AGE_AND_DECLINE,REFRESH,181+,304,down,-100.0,0.00,8.0
7,8,content_ccfb4d0227b1,20.033333,AGE_AND_DECLINE,REFRESH,181+,301,down,-100.0,0.00,10.8
8,9,content_02b0d6e30129,19.993333,AGE_AND_DECLINE,REFRESH,181+,313,down,-95.6,0.00,6.9
9,10,content_df1fa766cac2,19.903333,AGE_AND_DECLINE,REFRESH,181+,304,down,-97.7,0.49,4.8


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
# ML-07 — Top-20 review
# Review the highest-ranked rows using only observed signals.

top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = (
    "High priority because observed age and decline signals both support refresh."
)

top20["what_would_make_it_wrong"] = (
    "It could be wrong if the decline is temporary, the content is intentionally unchanged, "
    "or the observed signals do not reflect the current search opportunity."
)

top20_review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "score",
        "freshness_tier",
        "days_since_last_update",
        "trend_direction",
        "trend_pct",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

display(top20_review)

,rank,content_id,action,reason_code,score,freshness_tier,days_since_last_update,trend_direction,trend_pct,confidence_note,what_would_make_it_wrong
0,1,content_f6fdf87348f6,REFRESH,AGE_AND_DECLINE,22.433333,181+,373,down,-100.0,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."
1,2,content_1b4ec72dafd4,REFRESH,AGE_AND_DECLINE,22.400000,181+,372,down,-100.0,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."
2,3,content_55a5b1c46474,REFRESH,AGE_AND_DECLINE,21.283333,181+,373,down,-88.5,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."
3,4,content_7a888d3d99c8,REFRESH,AGE_AND_DECLINE,20.433333,181+,313,down,-100.0,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."
4,5,content_94991fe6268c,REFRESH,AGE_AND_DECLINE,20.433333,181+,313,down,-100.0,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."
5,6,content_ab18b5811c02,REFRESH,AGE_AND_DECLINE,20.166667,181+,305,down,-100.0,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."
6,7,content_84d12054c0c0,REFRESH,AGE_AND_DECLINE,20.133333,181+,304,down,-100.0,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."
7,8,content_ccfb4d0227b1,REFRESH,AGE_AND_DECLINE,20.033333,181+,301,down,-100.0,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."
8,9,content_02b0d6e30129,REFRESH,AGE_AND_DECLINE,19.993333,181+,313,down,-95.6,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."
9,10,content_df1fa766cac2,REFRESH,AGE_AND_DECLINE,19.903333,181+,304,down,-97.7,High priority because observed age and decline...,"It could be wrong if the decline is temporary,..."


In [19]:
# ML-07 — Signal audit
# Inspect the real signals we can use for the baseline rule.

signals = [
    "freshness_tier",
    "position_tier",
    "trend_direction",
    "impression_tier",
]

for signal in signals:
    print("\n" + "=" * 70)
    print(f"SIGNAL: {signal}")
    print("=" * 70)

    summary = (
        df.groupby(signal, dropna=False)
          .agg(
              n=("content_id", "size"),
              median_ctr=("ctr", "median"),
              median_position=("avg_position", "median"),
              median_search_volume=("search_volume", "median"),
              median_trend=("trend_pct", "median")
          )
          .sort_values("n", ascending=False)
    )

    display(summary)


SIGNAL: freshness_tier


,n,median_ctr,median_position,median_search_volume,median_trend
freshness_tier,,,,,
0-30,20480,0.04,9.9,10.0,-33.3
91-180,9171,0.10,13.6,10.0,-34.4
31-90,175,0.00,13.9,10.0,-38.5
181+,174,0.00,7.0,0.0,-44.5



SIGNAL: position_tier


,n,median_ctr,median_position,median_search_volume,median_trend
position_tier,,,,,
page_1,11814,0.16,6.6,10.0,-34.10
striking,7304,0.11,13.9,10.0,-34.90
page_3_5,7242,0.03,28.9,10.0,-32.20
top_3,2321,0.00,0.0,10.0,-52.25
deep,1319,0.00,61.0,10.0,1.20



SIGNAL: trend_direction


,n,median_ctr,median_position,median_search_volume,median_trend
trend_direction,,,,,
down,16262,0.08,11.3,10.0,-55.60
stable,5962,0.17,11.4,10.0,-3.80
up,4388,0.09,15.3,10.0,62.55
new,2236,0.00,0.0,10.0,NaN
flat,1152,0.00,6.6,10.0,NaN



SIGNAL: impression_tier


,n,median_ctr,median_position,median_search_volume,median_trend
impression_tier,,,,,
low,11248,0.00,9.1,10.0,-47.1
moderate,10469,0.12,14.1,10.0,-34.6
good,7205,0.21,9.4,10.0,-28.2
excellent,1078,0.22,6.5,10.0,-17.3


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [20]:
# ML-07 — Weak picks + leakage check

print("WEAK PICKS")
print("=" * 70)

# Show the lowest-ranked items as weak picks.
weak_picks = baseline_queue.tail(10).copy()

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "freshness_tier",
            "days_since_last_update",
            "trend_direction",
            "trend_pct",
        ]
    ]
)

print("\nLEAKAGE CHECK")
print("=" * 70)

# These are the columns used by the baseline rule.
rule_inputs = [
    "days_since_last_update",
    "trend_pct",
]

print("Rule inputs:")
for col in rule_inputs:
    print(f" - {col}")

# Confirm that no future-window or label-derived fields are used.
forbidden_terms = ["label", "future", "target", "outcome"]

used_columns = set(rule_inputs + ["score", "reason_code", "action"])

leakage_hits = [
    col for col in used_columns
    if any(term in col.lower() for term in forbidden_terms)
]

if leakage_hits:
    print("\nPotential leakage fields found:", leakage_hits)
else:
    print("\nLeakage check: PASS")
    print("No future-window or label-derived inputs are used by the baseline rule.")

WEAK PICKS


,rank,content_id,score,reason_code,action,freshness_tier,days_since_last_update,trend_direction,trend_pct
29990,29991,content_190cbb83cea5,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,flat,NaN
29991,29992,content_262ae9dd701a,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,up,200.0
29992,29993,content_00dda64bb29a,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,new,NaN
29993,29994,content_8bce3371c63c,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,flat,NaN
29994,29995,content_7a683a71e550,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,up,200.0
29995,29996,content_2a843f006d86,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,flat,NaN
29996,29997,content_3a8f5c52b1a0,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,flat,NaN
29997,29998,content_94283b065b7c,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,new,NaN
29998,29999,content_9ffe1e2e3575,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,flat,NaN
29999,30000,content_0a22a2eeefdd,0.033333,AGE_AND_DECLINE,REFRESH,0-30,1,new,NaN



LEAKAGE CHECK
Rule inputs:
 - days_since_last_update
 - trend_pct

Leakage check: PASS
No future-window or label-derived inputs are used by the baseline rule.


In [21]:
print("\n" + "=" * 70)
print("SIGNAL: freshness_tier")
print("=" * 70)

summary = (
    df.groupby("freshness_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          median_position=("avg_position", "median"),
          median_search_volume=("search_volume", "median"),
          median_days_since_update=("days_since_last_update", "median"),
          median_trend=("trend_pct", "median")
      )
      .sort_values("n", ascending=False)
)

display(summary)











# ML-07 signal verdicts
# Verdicts are based on the bucket tables above.

signal_verdicts = pd.DataFrame([
    {
        "signal": "freshness_tier",
        "verdict": "CONFIRMED",
        "rule": "Older/stale content is a stronger refresh candidate."
    },
    {
        "signal": "trend_direction",
        "verdict": "CONFIRMED",
        "rule": "Down-trending content is a stronger refresh candidate."
    }
])

display(signal_verdicts)


SIGNAL: freshness_tier


,n,median_ctr,median_position,median_search_volume,median_days_since_update,median_trend
freshness_tier,,,,,,
0-30,20480,0.04,9.9,10.0,20.0,-33.3
91-180,9171,0.10,13.6,10.0,104.0,-34.4
31-90,175,0.00,13.9,10.0,41.0,-38.5
181+,174,0.00,7.0,0.0,211.0,-44.5


,signal,verdict,rule
0,freshness_tier,CONFIRMED,Older/stale content is a stronger refresh cand...
1,trend_direction,CONFIRMED,Down-trending content is a stronger refresh ca...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.